In [100]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('data/temp/chats_edu.csv')

# quero que fique apenas as colunas relevantes
df = df[['session_id','ESCUELA', 'ESTADO', 'SEXO']]

# Remover linhas com valores nulos
df = df.dropna()

# Calcular contagem de mensagens por sessão
df['message_count'] = df.groupby('session_id')['session_id'].transform('count')

# Mostrar estatísticas da contagem de mensagens
print("Estatísticas de contagem de mensagens:")
print(df.groupby('session_id')['message_count'].first().describe())
print("\n")

# agrupar para deixar apenas uma linha por session_id
df = df.groupby('session_id').agg({
    'ESCUELA': 'first',
    'ESTADO': 'first',
    'SEXO': 'first',
    'message_count': 'first'
}).reset_index()

# Transformar as colunas categóricas em formato binário (one-hot encoding)
df_encoded = pd.get_dummies(df.drop(columns=['session_id', 'message_count']))

# 1. Encontrar itemsets frequentes
frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)

# 2. Gerar regras com base nos itemsets
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

# 3. Ver as colunas principais
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])





Estatísticas de contagem de mensagens:
count    2748.000000
mean        1.900291
std         2.131180
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        50.000000
Name: message_count, dtype: float64


                                          antecedents         consequents  \
0      (ESCUELA_ESCUELA DE ADMINISTRACIÓN Y NEGOCIOS)          (ESTADO_A)   
1   (ESCUELA_ESCUELA DE DESARROLLO SOCIAL Y SERVIC...          (ESTADO_A)   
2                      (ESCUELA_ESCUELA DE EDUCACIÓN)          (ESTADO_A)   
3                      (ESCUELA_ESCUELA DE EDUCACIÓN)            (SEXO_F)   
4                     (ESCUELA_ESCUELA DE TECNOLOGÍA)          (ESTADO_A)   
5                     (ESCUELA_ESCUELA DE TECNOLOGÍA)            (SEXO_M)   
6                                            (SEXO_F)          (ESTADO_A)   
7                                            (SEXO_M)          (ESTADO_A)   
8            (ESTADO_A, ESCUELA_ESCUELA DE EDUCACIÓN)         